# Week 7
## E-commerce Orders Delta Pipeline

In [0]:
import yaml
local_config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/E-commerce-Orders-Delta-Pipeline/config/projects_config.yml"

with open(local_config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
schema_name = config["schema"]   
landing_volume = config["landing_volume"]
checkpoint_volume = config["checkpoint_volume"]

landing_path = config["paths"]["landing_orders"]
checkpoint_path = config["paths"]["autoloader_checkpoint"]

bronze_table = config["tables"]["bronze_orders"]


print(f"landing_path: {landing_path}")
print(f"checkpoint_path: {checkpoint_path}")
print(f"bronze_table: {bronze_table}")


## Step - 2 Land the first raw file in the volume

In [0]:
dbutils.fs.mkdirs(landing_path)

repo_sample_path = "file:/Workspace/Repos/adb-tetiana@startsteps.org/E-commerce-Orders-Delta-Pipeline/sample_data/"

dbutils.fs.cp(
    f"{repo_sample_path}/batch_1_orders.csv",
    f"{landing_path}/batch_1_orders.csv"
)    

display(dbutils.fs.ls(landing_path))


In [0]:
raw_preview_df = (spark.read
                    .option("header", True)
                    .option("inferSchema", True)
                    .csv(f"{landing_path}/batch_1_orders.csv")
)                    


## Step -3 Use Auto Loader to ingest incrementaly into Delta

In [0]:
# defined the injections source
orders_stream = (
    spark.readStream
    .format("cloudFiles") # format files for autoloader
    .option("cloudFiles.format","csv") # type of the loaded files
    .option("cloudFiles.schemaLocation", checkpoint_path + "/schema") # where to take source data 
    .option("header", "true") # tells autoloader that the first row in file  contains colum names
    .load(landing_path)
)

# defines the output behaviour
query = (
    orders_stream.writeStream
    .option("checkpointLocation", checkpoint_path + "/schema") # where to write to, remembers the injestion process 
    .trigger(availableNow=True) # define, often process files; availableNow - tell to process what currently avialiable in the source
    .toTable(bronze_table)
)

query.awaitTermination()

In [0]:
%sql
USE retail_dev.orders;

SELECT * FROM bronze_orders LIMIT 10